In [16]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/repor234/CBC-lab-example report.pdf
/kaggle/input/reports/CBC-sample-report-with-notes_0.pdf
/kaggle/input/reports/cbc-test-report.webp
/kaggle/input/reports/blood_count_dataset.csv


In [6]:
!apt-get install -y poppler-utils


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpoppler-dev libpoppler-private-dev libpoppler118
The following NEW packages will be installed:
  poppler-utils
The following packages will be upgraded:
  libpoppler-dev libpoppler-private-dev libpoppler118
3 upgraded, 1 newly installed, 0 to remove and 162 not upgraded.
Need to get 1,469 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler-private-dev amd64 22.02.0-2ubuntu0.12 [199 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler-dev amd64 22.02.0-2ubuntu0.12 [5,186 B]
Get:3 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpoppler118 amd64 22.02.0-2ubuntu0.12 [1,079 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched

In [7]:
!pip install pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 88.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 74.5 MB/s eta 0:00:00:00:01


**Milestone 2**

In [2]:
import os
import re
import json
import pandas as pd
import pytesseract
from PIL import Image
from pdf2image import convert_from_path

# ============================================================
# 1. REFERENCE RANGES (CBC)
# ============================================================
REFERENCE_RANGES = {
    "Hemoglobin": {"female": (12.0, 15.5), "male": (14.0, 18.0)},
    "White_Blood_Cells": {"general": (4000, 11000)},
    "Platelet_Count": {"general": (150000, 410000)},
    "Red_Blood_Cells": {"general": (4.5, 5.5)},
    "MCV": {"general": (80, 100)},
    "MCH": {"general": (27, 33)},
    "MCHC": {"general": (32, 36)},
    "RDW": {"general": (11.5, 14.5)}
}

# ============================================================
# 2. OCR REGEX PATTERNS
# ============================================================
CBC_PATTERNS = {
    "Hemoglobin": r"hemoglobin.*?([0-9]+\.?[0-9]*)",
    "Red_Blood_Cells": r"(total rbc count|rbc).*?([0-9]+\.?[0-9]*)",
    "White_Blood_Cells": r"(total wbc count|wbc).*?([0-9]{3,6})",
    "Platelet_Count": r"platelet count.*?([0-9]{5,6})",
    "MCV": r"mcv.*?([0-9]+\.?[0-9]*)",
    "MCH": r"mch.*?([0-9]+\.?[0-9]*)",
    "MCHC": r"(mchc|mean corpuscular hemoglobin concentration).*?([0-9]+\.?[0-9]*)",
    "RDW": r"rdw.*?([0-9]+\.?[0-9]*)"
}

# ============================================================
# 3. PARAMETER CLASSIFICATION (SAFE)
# ============================================================
def classify(param, value, gender="female"):
    if param not in REFERENCE_RANGES:
        return "Unknown"

    gender = gender.lower()
    ranges = REFERENCE_RANGES[param]

    if gender in ranges:
        low, high = ranges[gender]
    elif "general" in ranges:
        low, high = ranges["general"]
    else:
        return "Unknown"

    if value < low:
        return "Low"
    elif value > high:
        return "High"
    return "Normal"

# ============================================================
# 4. CSV PARSER
# ============================================================
def parse_csv(file_path):
    return pd.read_csv(file_path)

# ============================================================
# 5. JSON PARSER
# ============================================================
def parse_json(file_path):
    with open(file_path) as f:
        data = json.load(f)
    return pd.DataFrame(data)

# ============================================================
# 6. PDF / IMAGE PARSER
# ============================================================
def parse_pdf_image(file_path):
    records = []
    images = convert_from_path(file_path) if file_path.lower().endswith(".pdf") else [Image.open(file_path)]

    for img in images:
        text = pytesseract.image_to_string(img).lower()
        record = {}

        for param, pattern in CBC_PATTERNS.items():
            match = re.search(pattern, text, re.DOTALL)
            if match:
                record[param] = float(match.groups()[-1])

        age_match = re.search(r"age[:\s]+(\d+)", text)
        gender_match = re.search(r"(sex|gender)[:\s]+(male|female)", text)

        record["Age"] = int(age_match.group(1)) if age_match else None
        record["Gender"] = gender_match.group(2).capitalize() if gender_match else "Female"

        if len(record) > 2:
            records.append(record)

    return pd.DataFrame(records)

# ============================================================
# 7. STANDARDIZATION + INTERPRETATION
# ============================================================
def standardize_and_classify(df):
    output = []

    for _, row in df.iterrows():
        gender = str(row.get("Gender", "Female")).capitalize()

        patient = {
            "Age": row.get("Age"),
            "Gender": gender,
            "Parameters": {}
        }

        for param in REFERENCE_RANGES:
            if param in row and pd.notna(row[param]):
                value = float(row[param])
                patient["Parameters"][param] = {
                    "value": value,
                    "status": classify(param, value, gender)
                }

        output.append(patient)

    return output

# ============================================================
# 8. PROCESS FILE
# ============================================================
def process_file(file_path):
    if file_path.endswith(".csv"):
        df = parse_csv(file_path)
    elif file_path.endswith(".json"):
        df = parse_json(file_path)
    elif file_path.lower().endswith((".pdf", ".png", ".jpg", ".jpeg", ".webp")):
        df = parse_pdf_image(file_path)
    else:
        raise ValueError("Unsupported file format")

    if df.empty:
        raise ValueError("No valid patient records found")

    return standardize_and_classify(df)

# ============================================================
# 9. SAVE OUTPUT (JSON + CSV)
# ============================================================
def save_outputs(results, base_name):
    with open(f"{base_name}.json", "w") as f:
        json.dump(results, f, indent=4)

    rows = []
    for patient in results:
        for param, details in patient["Parameters"].items():
            rows.append({
                "Age": patient["Age"],
                "Gender": patient["Gender"],
                "Parameter": param,
                "Value": details["value"],
                "Status": details["status"]
            })

    pd.DataFrame(rows).to_csv(f"{base_name}.csv", index=False)

# ============================================================
# 10. MODEL 2 — PATTERN RECOGNITION
# ============================================================
def analyze_patterns(patient):
    params = patient["Parameters"]
    findings, risks = [], []

    if "Hemoglobin" in params and "MCV" in params:
        hb, mcv = params["Hemoglobin"]["value"], params["MCV"]["value"]

        if hb < 12 and mcv < 80:
            findings.append("Microcytic anemia pattern detected.")
            risks.append("Anemia")
        elif hb < 12 and mcv > 100:
            findings.append("Macrocytic anemia pattern detected.")
            risks.append("Anemia")

    if "White_Blood_Cells" in params and params["White_Blood_Cells"]["value"] > 11000:
        findings.append("Possible infection or inflammation.")
        risks.append("Infection")

    if "Platelet_Count" in params and params["Platelet_Count"]["value"] < 150000:
        findings.append("Low platelet count – bleeding risk.")
        risks.append("Bleeding Risk")

    return {"Findings": findings, "Risk_Flags": list(set(risks))}

# ============================================================
# 11. MODEL 3 — CONTEXTUAL ANALYSIS
# ============================================================
def contextual_analysis(patient, model2):
    refined = model2["Findings"].copy()
    age = patient.get("Age")

    if age and age > 60 and model2["Risk_Flags"]:
        refined.append("Advanced age increases clinical relevance.")

    return {"Findings": refined, "Risk_Flags": model2["Risk_Flags"]}

# ============================================================
# 12. FINAL REPORT
# ============================================================
def generate_final_report(patient):
    model2 = analyze_patterns(patient)
    model3 = contextual_analysis(patient, model2)

    recommendations = (
        ["No major abnormalities detected. Maintain a healthy lifestyle."]
        if not model3["Risk_Flags"]
        else ["Consult a physician for further evaluation."]
    )

    return {
        "Patient_Info": {"Age": patient["Age"], "Gender": patient["Gender"]},
        "Parameter_Analysis": patient["Parameters"],
        "Findings": model3["Findings"],
        "Risk_Flags": model3["Risk_Flags"],
        "Recommendations": recommendations,
        
    }

# ============================================================
# 13. ORCHESTRATOR
# ============================================================
def run_milestone_2(results):
    return [generate_final_report(p) for p in results]

# ============================================================
# 14. RUN
# ============================================================
if __name__ == "__main__":
    file_path = "/kaggle/input/reports/blood_count_dataset.csv"

    results = process_file(file_path)
    save_outputs(results, "/kaggle/working/milestone1_output")

    final_reports = run_milestone_2(results)

    with open("/kaggle/working/final_health_report.json", "w") as f:
        json.dump(final_reports, f, indent=4)

    print(json.dumps(final_reports, indent=4))


[
    {
        "Patient_Info": {
            "Age": 68,
            "Gender": "Female"
        },
        "Parameter_Analysis": {
            "Hemoglobin": {
                "value": 10.4,
                "status": "Low"
            },
            "White_Blood_Cells": {
                "value": 5700.0,
                "status": "Normal"
            },
            "Platelet_Count": {
                "value": 180000.0,
                "status": "Normal"
            },
            "Red_Blood_Cells": {
                "value": 3.7,
                "status": "Low"
            },
            "MCV": {
                "value": 77.0,
                "status": "Low"
            },
            "MCH": {
                "value": 25.0,
                "status": "Low"
            },
            "MCHC": {
                "value": 32.0,
                "status": "Normal"
            }
        },
        "Findings": [
            "Microcytic anemia pattern detected.",
            "Advanced age increase